# TP 11 – Le modèle de pourboire en temps réel, puis sous le capot (55 min)

**Objectif** : appliquer sur le flux de courses le modèle entraîné au TP 7, puis lire et comparer des plans d'exécution.

**Réussi si** : des prédictions sont produites en continu et vous savez expliquer le plan.

**Avant de commencer**, dans un terminal à la racine du dépôt :

```bash
python scripts/stream_simulator.py courses --reset          # en direct : un fichier toutes les 2 secondes
# ou, si vous êtes en retard / en mode dégradé :
python scripts/stream_simulator.py courses --all --reset    # tout d'un coup
```

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("TP").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")     # petit volume : 200 partitions par défaut, c'est trop

from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
import time
zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True).select(
    F.col("LocationID").alias("PULocationID"), "Borough")

### 1. La préparation des features, dans UNE fonction (la même qu'au TP 7)

Carte uniquement, `heure`, `manhattan`, `nuit`, `label` (pourboire > 0).

In [ ]:
def features(df):
    return (df.filter("trip_distance > 0 AND payment_type = 1")
              ...)

### 2. Recharger le modèle du TP 7 : `PipelineModel.load("out/modele_pourboire")`

S'il n'existe pas, lancez d'abord `python jour2/corriges/tp7_classification.py`.

In [ ]:
modele = ...

### 3. Appliquer le modèle sur le flux

`modele.transform(features(flux))`, puis `p_pourboire` = `vector_to_array("probability")[1]`.

In [ ]:
schema = spark.read.parquet("data/trips.parquet").schema
flux = ...
predictions = ...

### 4. Taux de pourboire **prédit** (moyenne de `p_pourboire`) vs **réel** (moyenne de `label`), par fenêtre de 30 min et par `Borough`

Watermark de 10 minutes sur `ts` = `tpep_pickup_datetime` converti en timestamp. Pourquoi la moyenne des probabilités et pas des prédictions 0/1 ?

In [ ]:
suivi = ...

### 5. `foreachBatch` : pour chaque micro-batch, écrire en Parquet (`out/tp11/suivi`) et afficher le nombre de fenêtres

In [ ]:
def traiter(batch_df, batch_id):
    ...

q = ...

### 6. Sous le capot : `explain("formatted")` sur la même requête, en CSV puis en Parquet

```python
requete = lambda df: df.filter("trip_distance > 10").groupBy("PULocationID").agg(F.avg("total_amount"))
```
Comparez `PushedFilters`, `ReadSchema`, `Batched`. Mesurez le temps de chaque version (3 exécutions).

In [ ]:
...

### 7. AQE : exécutez la requête Parquet, puis `explain()`. Trouvez `isFinalPlan=true` et `AQEShuffleRead`. Combien de blocs `*(n)` (whole-stage codegen) ?

In [ ]:
...

### Bonus – Spark UI, onglet **Structured Streaming** : débit, durée des micro-batches, lignes d'état